# 10a · 数据湖和 Athena：用 SQL 查 2.18 亿个分数

## 这一章在整个项目里的位置

```
00–09 ✅ → [10 对外服务] → 11 RAG → 12 Agent → 13 收尾
              │
              ├─ 10a 数据湖 + Athena：分数导出到 AWS，用 SQL 查   ← 你在这里
              └─ 10b API：FastAPI → 容器 → Lambda → API Gateway
```

## 背景：为什么需要这一章

09c 结束时，2.18 亿个分数在 Databricks 的表里（`gold.fct_variant_effect`）。但是：
- **Databricks 是工程师干活的地方**，不是对外服务的地方。公司里对外的服务、给别的团队用的数据，一般放在**云**上（我们是 AWS），这是 2026-10-07 定稿时的第 6 条决定
- 10b 要做一个 API，任何人问"TP53 R175H 的分数是多少"，一两秒内回答。它需要一个**按次付费、不用一直开着**的地方去查

这一章把分数搬到 AWS 的**数据湖**（S3 里的文件），再用 **Athena** 直接对这些文件写 SQL。

## 你在哪里操作

```
你：在这个 notebook 里运行一格
   ├─ 第 2 步：Terraform 在 AWS 上登记表结构（1 分钟）
   ├─ 第 3 步：Databricks 把分数写成文件，传到 S3（约 5 分钟）
   └─ 第 6 步：用 SQL 查
看结果：AWS 控制台 → Athena → 查询编辑器（右上角工作组选 qafi-dev）
```

## 这一章做的 6 件事

| # | 做什么 | 比喻 |
|---|---|---|
| 1 | 为什么是 S3 + Athena | 选仓库 |
| 2 | 登记表结构（Glue 目录） | 给仓库建一本目录 |
| 3 | 导出：Databricks → S3 | 按编号装箱，送进仓库 |
| 4 | 核对 | 对着装箱单清点 |
| 5 | 发布 | 在门口挂牌"现在卖的是这一批" |
| 6 | 用 SQL 查 | 按目录取货，只开需要的那个箱子 |

**花费**：几美分（存储每月约 0.1 美元，每次查询不到 0.01 美分）。**时间**：约半小时。

**开始前**：在终端运行 `aws login` 登录 AWS（00 号学过）。

---

## 准备

运行下面这一格。**不用看代码。**

In [ ]:
import json, os, subprocess, time, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=FutureWarning)
import boto3
import pandas as pd
from botocore.config import Config
from databricks.sdk import WorkspaceClient
from qafi_platform import lake
from qafi_platform.lakehouse import sql
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ["AWS_PROFILE"] = "qafi"   # Terraform、aws 命令和 Python 都用 00 号登录的身份
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
aws = boto3.Session(region_name="eu-central-1")
s3 = aws.client("s3", config=Config(signature_version="s3v4", s3={"addressing_style": "virtual"}))
athena_client, w = aws.client("athena"), WorkspaceClient(profile="qafi")
CATALOG = "qafi_dev"
print("AWS:", aws.client("sts").get_caller_identity()["Arn"], "| Databricks:", w.config.host)

---

## 第 1 步 · 为什么是 S3 + Athena

### 在干什么
弄清楚分数放在哪里、用什么去查，以及为什么这样选。

### 背景
比喻：**选仓库**。我们的数据有三个特点：
1. **很大，但不变**：2.18 亿行，一个版本算好就不再改
2. **查询很简单**：几乎都是"某个蛋白、某个位置"
3. **查得不多**：一天可能几十次，不是每秒几千次

候选方案：

| 方案 | 是什么 | 一直开着要花钱吗 | 一次查询多快 | 适合我们吗 |
|---|---|---|---|---|
| 留在 Databricks | 数据平台的 SQL 仓库 | 查询时要先启动仓库 | 几秒 | 对外服务不放在数据平台上 |
| 关系数据库（RDS） | 传统数据库 | **要**，每月几十美元 | 毫秒 | 太贵 |
| DynamoDB | 键值数据库 | 不用（按次付费） | 毫秒 | 可以，但要把 2.18 亿行导进去，也不能随手写 SQL |
| **S3 + Athena** | 文件放在 S3，用 SQL 直接查文件 | **不用**：存储几美分，查询按扫描量付费 | 1–2 秒 | **选它** |

**取舍**：Athena 每次查询要 1–2 秒，比数据库慢。对一个"偶尔查一下"的服务，这完全够用，换来的是**不查就不花钱**。如果以后变成每秒几千次查询，再把热门数据放进 DynamoDB，这是公司里常见的升级路线。

### 用了什么技术，为什么
- **S3**：AWS 的对象存储，就是放文件的地方。01 号建的 `lake` 桶就是为这一章准备的
- **Parquet**：按列存储的文件格式（04 号起一直在用）。查询只读需要的列
- **Glue Data Catalog**：AWS 的"表目录"。它不存数据，只记录"这张表有哪些列、文件在 S3 哪里"
- **Athena**：AWS 的无服务器 SQL 引擎（底层是开源的 Trino）。没有要开的机器，写 SQL 就查，按扫描的数据量付费：**每 TB 5 美元**

### 公司里什么情况怎么做
- "**数据湖**"就是这个组合：文件放在对象存储里，目录记表结构，用 SQL 引擎直接查文件。数据平台（Databricks）负责算，数据湖负责存和分发
- 选存储先看**查询模式**：查得多、要毫秒级用数据库；查得少、要分析用数据湖

### 面试怎么说
> "The scores are computed once per release and read rarely, so I serve them from S3 with Athena: no always-on database, storage costs cents, and each lookup costs a fraction of a cent. The trade-off is one to two seconds per query; if traffic grew, hot keys would move to DynamoDB."

---

## 第 2 步 · 登记表结构（Glue 目录）

### 在干什么
用 Terraform 在 AWS 上建 Glue 目录里的两张表，以及 Athena 的工作组。

### 背景
比喻：**给仓库建一本目录**。仓库（S3）里只有文件。目录（Glue）上写着：
- `variant_effect` 表：12 列（蛋白编号、基因名、位置、两个分数……），文件在 `s3://…-lake/variant_effect/`，格式 Parquet
- `proteins` 表：每个蛋白一行（基因名、长度……）

**先有文件还是先有目录都行**：Athena 查询时才按目录去读文件。这叫 **schema on read**（读的时候才套上表结构），和数据库"写进去就得符合表结构"相反。

**分区**（partition）：每张表按 `release`（版本）分区，每个版本一个文件夹：

```
s3://…-lake/variant_effect/release=2026-10-14/part-00.parquet ...
s3://…-lake/variant_effect/release=2026-11-20/...      ← 下一版，和上一版并排放着
```

查询写上 `WHERE release = '…'`，Athena 就只看那一个文件夹。

**工作组**（workgroup）`qafi-dev`：所有查询都在这里跑，有三条规定：
- 查询结果存到 `athena` 桶，7 天后自动删除，并用平台的密钥加密
- **一次查询最多扫 5 GB**，超过就自动停止。最坏的情况一次也只花 2.5 美分。这是一道**成本护栏**
- 这些规定是强制的，查询的人改不了

### 执行
这一次 Terraform 也会顺便建好 10b 要用的镜像仓库、API 的身份和日志组：它们是空的，不花钱。

In [ ]:
run("terraform -chdir=infra/dev init -input=false -no-color")
run("set -o pipefail; terraform -chdir=infra/dev plan -input=false -no-color -out=dev.tfplan 2>&1 | grep -E 'will be|Plan:|No changes|Error'")

### 结果怎么看
最后一行 `Plan: 11 to add, 0 to change, 0 to destroy`（如果这些资源已经建好了，比如第二次运行，会显示 `No changes`，同样没问题）：
- `aws_glue_catalog_database.lake`、`aws_glue_catalog_table.lake["variant_effect"]`、`["proteins"]`：目录和两张表
- `aws_athena_workgroup.lake`：工作组
- `aws_ecr_repository.api`、`aws_iam_role.api` 等：10b 用的，先建好

**0 to change, 0 to destroy** 很重要：已有的桶、函数一个都不动。确认后执行：

In [ ]:
run("set -o pipefail; terraform -chdir=infra/dev apply -input=false -no-color dev.tfplan 2>&1 | grep -E 'complete|Error'")
LAKE = json.loads(subprocess.run("terraform -chdir=infra/dev output -json lake", shell=True, capture_output=True, text=True).stdout)
LAKE

### 去网页上看
AWS 控制台 → **Glue** → Data Catalog → Databases → `qafi_dev_lake`：两张表，点开能看到每一列和它的说明。

### 公司里什么情况怎么做
- 表结构写在 Terraform 里，和代码一起审核、一起版本管理。不要在网页上手动建表
- 每个团队一个工作组，各自有扫描上限和费用统计

### 面试怎么说
> "The Glue Data Catalog holds the table definitions, written in Terraform; the data stays in S3 and is read with schema on read. An Athena workgroup enforces the result location, encryption and a 5 GB scan limit per query as a cost guardrail."

---

## 第 3 步 · 导出：Databricks → S3

### 在干什么
让 Databricks 把这一版的分数写成文件，直接传进 S3。

### 背景
比喻：**按编号装箱，送进仓库**。有两个问题要解决。

**问题一：Databricks 怎么往我们的桶里写？**
公司里，Databricks 会被授权直接写 S3（一个 IAM 角色加上 Unity Catalog 的 "external location"）。免费版不支持。所以用 04 号进货时的办法，只是方向反过来：
- 04 号：你签**临时下载链接**，Databricks 从 S3 读
- 这里：你签**临时上传链接**，Databricks 往 S3 写

每个链接只能往**一个指定的文件名**写，**6 小时**后失效。Databricks 手里没有任何 AWS 密钥。这叫**最小权限**：给的权限刚好够做这件事，多一点都不给。

**链接本身就是钥匙**：拿到链接的人在 6 小时内都能往那个文件名写。所以代码只打印文件名，不打印链接。

**问题二：怎么装箱，查一个蛋白才快？**
2.18 亿行，如果随便装，查 TP53 就得把所有箱子都打开看一遍（扫描几个 GB）。我们这样装：
1. 按**蛋白编号排好序**，切成 **16 个文件**。每个文件装一段连续的编号（比如 A0A…到 O14…）
2. 每个文件里再分成约 16 MB 的**小段**（row group）
3. Parquet 文件的末尾有一张"目录"：每一小段**最小和最大的蛋白编号**

查 `uniprot_id = 'P04637'` 时，Athena 先读每个文件末尾的目录（很小），发现只有一个文件的一小段可能有 P04637，**只读那一小段**。这叫**按统计信息跳过**（predicate pushdown）。

**一个版本只写一次**：如果 S3 里已经有这一版，代码会拒绝再写（除非你明确要覆盖）。发布出去的东西不能悄悄改。

**装箱单最后写**：所有文件都传完，最后才传一份清单（manifest）：每个文件多少行、多大。清单在，就说明这一版完整。

### 用了什么技术，为什么
- **预签名上传链接**（presigned PUT URL）：S3 的标准功能，用你的身份签名，限定一个文件名和有效期
- **Spark 的 `repartitionByRange` + `sortWithinPartitions`**：按编号切段、段内排序
- 新的 Databricks 任务 **`qafi lake export`**：读 `gold.fct_variant_effect`，写文件，上传

### 执行
先签链接，放进 Databricks 的存储区：

In [ ]:
releases = sql(w, f"SELECT DISTINCT release FROM {CATALOG}.ml.proteome_scores")["release"].tolist()
assert len(releases) == 1, releases
RELEASE = releases[0]
plan = lake.upload_plan(s3, LAKE["bucket"], RELEASE)
print(f"版本 {RELEASE}：签好 {len(plan['variant_effect']) + 2} 个上传链接，6 小时内有效")
print("例如这个文件名：", plan["variant_effect"][0]["key"])
print("链接清单放在：", lake.upload_plan_to_volume(w, CATALOG, plan))

再部署并运行导出任务（约 5 分钟，这一格会一直等到它结束）：

In [ ]:
run("cd databricks && databricks bundle deploy")
started = time.time()
run("cd databricks && databricks bundle run lake_export")
print(f"用了 {(time.time() - started) / 60:.0f} 分钟")

### 结果怎么看
最后几行是任务的输出：每个文件上传了多少行、多少 MB，最后一行 `release …: 2.18 亿 variants of 20,652 proteins in S3`。

如果这一格报错 `already in s3`：说明这一版已经导出过了，直接往下走第 4 步核对就行。

### 公司里什么情况怎么做
- 跨平台交数据，用**临时、限定范围的凭证**，不交长期密钥
- 发布的数据**不可变**：一个版本一个文件夹，有问题就发新版本，不改旧的
- 数据的**排序和切分**按最常见的查询来设计：我们最常按蛋白查，就按蛋白排序

### 面试怎么说
> "Free Edition cannot hold AWS credentials, so the laptop signs one presigned PUT URL per output file, valid for six hours; the Databricks job writes the release range-partitioned and sorted by protein, uploads the files and the manifest last. Each release is an immutable folder."

---

## 第 4 步 · 核对

### 在干什么
对着清单检查 S3 里的每一个文件。

### 背景
比喻：**对着装箱单清点**。检查三件事：
- 清单上的每个文件都在 S3 里
- 每个文件的大小和清单上一样（没有传一半的）
- S3 里没有清单之外的文件（比如上一次失败留下的）

### 执行

In [ ]:
files = lake.check(s3, LAKE["bucket"], RELEASE)
print(f"{files.attrs['variants']:,} 个突变，{files.attrs['proteins']:,} 个蛋白，{files['bytes'].sum() / 1e9:.2f} GB；模型版本 {files.attrs['model_versions']}")
files[["key", "rows", "bytes", "row_groups", "first_protein", "last_protein", "ok"]]

### 结果怎么看
第一行：2.18 亿个突变、20,652 个蛋白、2.64 GB，以及打分用的模型版本（和 09b 注册的冠军版本一致）。

下面的表一共 17 行：16 个 `variant_effect` 文件和 1 个 `proteins` 文件。写这一章时：
- `rows`、`bytes`：每个文件约 1,360 万行、约 165 MB（按范围切，各段不完全一样大）
- `row_groups`：每个文件 18–19 小段，每小段约 70 万行、约 9 MB，大约 60 个蛋白
- `first_protein`、`last_protein`：每个文件装的蛋白编号范围，一个接一个，互不重叠（A0A024R1R8 到 O14893，O14894 到 O94927……）。这就是"按编号排好序、切成 16 段"
- `ok` 全是 True：S3 里的文件和清单一模一样，没有多也没有少

---

## 第 5 步 · 发布

### 在干什么
告诉目录"这一版来了"，并把"当前版本"的牌子挂到这一版上。

### 背景
比喻：**在门口挂牌**"现在卖的是这一批"。两件事：
1. **登记分区**：在目录里加一行"`release = 2026-10-15` 的文件在这个文件夹"
2. **挪牌子**：两个**视图**（view）`variant_effect_current` 和 `proteins_current`，内容是"`release` 等于这一版的那些行"

API（10b）只查这两个视图，从不直接查表。所以：
- **发布新版本** = 把视图指向新版本
- **回滚** = 把视图指回旧版本

这和 09b 模型注册表里的 `champion` 标签是同一个思路：**用的人只认标签**。模型有标签，数据也有标签。

### 执行

In [ ]:
lake.publish(athena_client, LAKE["workgroup"], LAKE["database"], LAKE["bucket"], RELEASE)
q = lambda statement: lake.query(athena_client, statement, LAKE["workgroup"], LAKE["database"])
q("SELECT release, count(*) AS proteins FROM proteins_current GROUP BY release")

### 结果怎么看
一行：这一版，20,652 个蛋白。从现在起，查 `*_current` 就是查这一版。

### 面试怎么说
> "Each release is registered as a partition, and two views point at the release being served. Releasing or rolling back data is redefining the views: the same alias pattern as the model registry."

---

## 第 6 步 · 用 SQL 查

### 在干什么
直接用 SQL 查数据湖，看看"只开需要的箱子"省了多少。

### 背景
问三个问题，看 Athena 各要读多少数据：
1. **TP53 第 175 位的 19 种替换**（按蛋白编号查）：文件按编号排过序，Athena 看每个文件末尾的目录，只读**一小段**
2. **所有蛋白里的 R175H**（按突变写法查）：哪个蛋白都可能有 R175H，和排序无关，**每一小段都得读**（但只读用到的几列）
3. **哪些基因里"两个分数都极低（< 0.05）"的突变最多**：全蛋白组的统计，要读全部 2.18 亿行，但只读其中 3 列

第 1 和第 2 个的差别就是**排序**的作用；第 3 个能做到只读 3 列，是**按列存储**的作用。

顺便一提：按基因名（`gene_name = 'TP53'`）查也一样快。一个蛋白内基因名都相同，Parquet 给每一小段存了一份"这一段有哪些基因名"的清单（字典），Athena 看一眼就知道要不要读。

### 执行

In [ ]:
tp53 = q("""SELECT variant, round(stability_score, 2) AS stability, round(function_score, 2) AS function
            FROM variant_effect_current WHERE uniprot_id = 'P04637' AND position = 175 ORDER BY alt_aa""")
r175h = q("""SELECT gene_name, round(stability_score, 2) AS stability, round(function_score, 2) AS function
             FROM variant_effect_current WHERE variant = 'R175H'""")
severe = q("""SELECT gene_name, count_if(stability_score < 0.05 AND function_score < 0.05) AS severe, count(*) AS variants
              FROM variant_effect_current GROUP BY gene_name ORDER BY severe DESC LIMIT 10""")
cost = lambda mb: f"{max(mb, 10) / 1e6 * 5 * 100:.4f} 美分"   # 5 美元 / TB，每次至少按 10 MB 算
display(pd.DataFrame({
    "查询": ["1. TP53 第 175 位（按编号）", "2. 所有蛋白里的 R175H", "3. 每个基因的极低分突变"],
    "找到几行": [len(tp53), len(r175h), len(severe)],
    "扫描 (MB)": [f.attrs["scanned_mb"] for f in (tp53, r175h, severe)],
    "秒": [f.attrs["seconds"] for f in (tp53, r175h, severe)],
    "花费": [cost(f.attrs["scanned_mb"]) for f in (tp53, r175h, severe)],
}))
display(tp53.head(5))
severe

### 结果怎么看
写这一章时的结果：

| 查询 | 扫描 | 花费 | 为什么 |
|---|---|---|---|
| 1. TP53 第 175 位 | **6.5 MB** | 0.005 美分 | 按编号排过序，只读一小段 |
| 2. 所有蛋白里的 R175H | 1,760 MB（**270 倍**） | 0.88 美分 | 和排序无关，"突变写法"这一列每一段都得读 |
| 3. 每个基因的极低分突变 | 1,535 MB | 0.77 美分 | 读全部 2.18 亿行，但只读 3 列 |

- 三个查询都在 **2 秒左右**：Athena 有很多机器同时读，读 6 MB 和读 1.7 GB 的时间差不多，**差的是钱**
- **花费**一列：Athena 每次查询至少按 10 MB 收费，所以小查询都是同一个最低价
- 第 3 个如果放在按行存储的数据库里，就得把 12 列全读一遍；按列存储只读用到的 3 列（全表 2.64 GB，这里读了 1.5 GB）

下面是 TP53 第 175 位的前 5 种替换（和 09c 第 4 步试跑的分数一致）。最后是"两个分数都极低（< 0.05）的突变"最多的 10 个基因：排第一的是 **TTN**（肌联蛋白），因为它是人体最大的蛋白（3.4 万个残基）。**这里数的是个数，大蛋白自然多**；想找"最敏感"的蛋白，应该看比例（`severe / variants`）。写 SQL 时，先想清楚问的是个数还是比例

### 去网页上看
AWS 控制台 → **Athena** → 查询编辑器（右上角工作组选 `qafi-dev`，数据库选 `qafi_dev_lake`）。随便写一句 SQL 试试，比如：

```sql
SELECT * FROM variant_effect_current WHERE uniprot_id = 'P38398' AND position = 1699
```
（BRCA1 第 1699 位）。下方会显示扫描了多少数据。

### 公司里什么情况怎么做
- 数据湖上的查询**按扫描量付费**，所以文件怎么排序、怎么切分，直接决定成本和速度。设计数据布局时，先问"大家最常怎么查"
- 分析师写 SQL 查数据湖，是公司里最常见的用法之一，不需要任何服务器

### 面试怎么说
> "Because the files are sorted by protein, a lookup by accession reads one row group: 6.5 MB of a 2.6 GB table. A filter on a column unrelated to the sort order read 270 times more. A proteome-wide aggregation reads only the columns it needs. In a lake, data layout decides the cost of every query."

---

## 提交代码

In [ ]:
run("git add -A")
run("uv run pre-commit run", check=False)
run("git add -A")
run('(git diff --cached --quiet || git commit -q -m "feat: 10a released scores in the AWS lake, Glue catalog and Athena") && git push -q')

推送后去 GitHub 的 Actions 页面看一眼，应该全绿。

---

## 本章小结

| 步骤 | 做了什么 | 记住什么 |
|---|---|---|
| 1 选型 | S3 + Athena，不用一直开着的数据库 | 按查询模式选存储 |
| 2 目录 | Glue 两张表，按版本分区；工作组有扫描上限 | 表结构写进 Terraform；成本护栏 |
| 3 导出 | 临时上传链接；按蛋白排序切成 16 个文件 | 最小权限；数据布局为查询服务 |
| 4 核对 | 清单对 S3，一个不多一个不少 | 清单最后写，在就说明完整 |
| 5 发布 | 分区 + `*_current` 视图 | 数据也有"冠军"标签，回滚就是挪标签 |
| 6 查询 | 按编号查只读一小段 | 跳过不需要的数据，又快又省钱 |

**一句话**：2.18 亿个分数从数据平台搬进了 AWS 数据湖，5 分钟导出；按蛋白查一次只读 6.5 MB，花最低价 0.005 美分。

## 面试题

1. **Why S3 and Athena instead of a database?**
   > "Batch-computed, read-mostly data with rare simple lookups: no always-on cost, SQL for analysts, cents per month. Latency of one to two seconds is acceptable; DynamoDB would be the next step for high traffic."

2. **How did you make lookups in 218 million rows cheap?**
   > "Range-partitioned and sorted by protein, 16 MB row groups, so Parquet min/max statistics let Athena skip everything but one row group; partitions by release; a 5 GB per-query cutoff as a guardrail."

3. **How do releases and rollbacks work for data?**
   > "Each release is an immutable S3 folder registered as a partition, with a manifest written last; views point at the served release, so releasing and rolling back is redefining a view."

4. **How do you move data between platforms without sharing long-lived keys?**
   > "Presigned URLs: scoped to one object and a few hours. In a full setup, a Unity Catalog external location with an IAM role."

**下一步 10b · API**：用 FastAPI 写一个查询接口，打包成容器，部署到 Lambda，通过 API Gateway 对外提供服务。